In [4]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb

import hashlib
# Check if CUDA is available and set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "all-MiniLM-L6-v2" 

model = SentenceTransformer(model_name)
model = model.to(device)


Using device: cuda


In [5]:
def get_chrome_client(collection_name):
    # Initialize ChromaDB client with updated configuration
    persist_directory = "/home/shubham/build_in_public/document-chatbot/pdf-chatbot-rag/chroma_db"
    chroma_client = chromadb.PersistentClient(path=persist_directory)
    print(f"ChromaDB client initialized with persistence at: {persist_directory}")
   
    try:
        collection = chroma_client.get_collection(name=collection_name)
        print(f"Using existing collection: {collection_name}")
    except:
        collection = chroma_client.create_collection(
            name=collection_name,
            metadata={"description": "Government of India Budget 2025-2026 documents"}
        )
        print(f"Created new collection: {collection_name}")
    return chroma_client, collection

collection_name = "budget_rag"
chroma_client, collection = get_chrome_client(collection_name)


ChromaDB client initialized with persistence at: /home/shubham/build_in_public/document-chatbot/pdf-chatbot-rag/chroma_db
Using existing collection: budget_rag


In [33]:
def process_chunk(chunks):
    for chunk in chunks:
        page_content = chunk["page_content"]

        metadata = {
            "chunk_id": chunk['chunk_id'],
            "prev_chunk_id": chunk.get("prev_chunk_id", None),
            "next_chunk_id": chunk.get("next_chunk_id", None),
            "page_id": chunk["page_id"],
            "title": chunk["metadata"]["title"],
            "author": chunk["metadata"]["author"],
            "file_path": chunk["metadata"]["file_path"],
            "page_num": chunk["metadata"]["page_num"],
            "total_pages": chunk["metadata"]["total_pages"],
            "chunk_index": chunk["chunk_index"],}
        
        if len(chunk["metadata"]) < 1 :
            metadata.update({'Header' : "None"})
        else: 
            metadata.update(chunk["metadata"]['headers'])

        embeddings_list = model.encode(page_content, convert_to_tensor=False).tolist()

        yield     {
            'id': chunk['chunk_id'], 
            'embedding' : embeddings_list,
            'metadata' : metadata, 
            'document'  : page_content
        }
    

In [38]:
import json 
with open("/home/shubham/build_in_public/document-chatbot_archive/pdf-chatbot-rag/chunks.json", 'r') as f: 
    chunks = json.load(f) 

In [54]:
def insert_batch(batch):
    """Helper function to insert a batch of chunks into ChromaDB."""
    ids = [chunk["id"] for chunk in batch]
    embeddings = [chunk["embedding"] for chunk in batch]
    metadatas = [chunk["metadata"] for chunk in batch]
    documents = [chunk["document"] for chunk in batch]

    # collection.add(embeddings=embeddings, metadatas=metadatas, ids=ids, documents=documents)

    print(f"Inserted {len(batch)} chunks into ChromaDB")


In [55]:
from itertools import islice

def batched(chunk_iter, batch_size):
    " Return the batch size until it's empty"
    for batch in iter(lambda: list(islice(chunk_iter, batch_size)), []):
        yield batch

def insert_chunks_in_batches(chunks, batch_size=100):
    """Insert chunks into ChromaDB."""
    for batch in batched(process_chunk(chunks), batch_size):
        insert_batch(batch)


In [56]:
insert_chunks_in_batches(chunks, batch_size=100)

Inserted 100 chunks into ChromaDB
Inserted 100 chunks into ChromaDB
Inserted 100 chunks into ChromaDB
Inserted 100 chunks into ChromaDB
Inserted 100 chunks into ChromaDB
Inserted 100 chunks into ChromaDB
Inserted 100 chunks into ChromaDB
Inserted 16 chunks into ChromaDB


In [ ]:
# processed_chunk

In [57]:

# collection.add(
#     documents=[processed_chunk["document"]],
#     embeddings=processed_chunk['embedding'],
#     metadatas=[processed_chunk['metadata']],
#     ids=[processed_chunk['id']]
# )

In [10]:
[processed_chunk['id']]

['5713f13aafdd2f73']

In [11]:
result = collection.get(
    ids=["5713f13aafdd2f73"],
    include=["documents", "embeddings", "metadatas"]
)
result

{'ids': ['5713f13aafdd2f73'],
 'embeddings': array([[-5.29075041e-02,  4.89339679e-02,  2.06608959e-02,
          2.95051932e-02,  2.82263830e-02, -4.05207369e-03,
          7.64205456e-02, -3.12054306e-02, -6.21808805e-02,
         -1.10190418e-02,  1.32264281e-02, -7.59976879e-02,
          6.75251111e-02,  2.61458866e-02, -1.64826512e-02,
         -7.09677041e-02, -7.90118240e-03, -1.07450806e-01,
          1.03786902e-03,  4.25277054e-02, -1.33315995e-01,
         -7.55208284e-02, -3.96361351e-02, -2.91165374e-02,
          1.68179534e-02,  6.09210283e-02, -8.13735873e-02,
          3.40429619e-02,  2.28178650e-02, -9.33419392e-02,
         -4.04625991e-03,  3.02131008e-02,  7.04288408e-02,
         -4.08664979e-02,  4.48659770e-02, -4.86471988e-02,
         -5.40167466e-02,  6.53706351e-03, -3.02589755e-03,
          2.42529295e-05, -1.85080823e-02, -5.41376583e-02,
         -8.81083030e-03,  1.93072716e-03, -2.25971993e-02,
         -7.08963946e-02, -7.72067606e-02, -2.59985905e-

In [12]:
total_chunks = len(chunks)
total_chunks


716

In [13]:
batch_size = 100 
for i in range(0, total_chunks, batch_size):
    
    print(len(chunks[i: i + batch_size]))

100
100
100
100
100
100
100
16


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def insert_chunk_into_db(chunks, batch_size=100):
    # create batches 
    total_chunks = len(chunks)
    for i in range(0, total_chunks, batch_size):
        batch = chunks[i: i + batch_size]

        with ThreadPoolExecutor() as executor:
            
            process_chunk(batch)

In [ ]:

# collection.add(
#     documents=[one_chunk["page_content"]],
#     embeddings=embeddings_list,
#     metadatas=[metadata],
#     ids=[one_chunk['chunk_id']]
# )

In [14]:
# chroma_client.delete_collection(name=collection_name) 

In [2]:
import json
with open("/home/shubham/build_in_public/pdf-chatbot/intermediate_data/chunks.json", 'r') as f: 
    data = json.load(f)


In [3]:
data[0]

{'chunk_id': '3453d1df95b70494',
 'prev_chunk_id': None,
 'next_chunk_id': '16af4eb4ed8e6882',
 'page_id': '26b3ea85-1eb2-49c0-9db0-1788779c3dcb',
 'metadata': {'title': '',
  'author': 'hss',
  'file_path': './data/budget_speech.pdf',
  'page_num': 1,
  'total_pages': 60,
  'headers': {}},
 'page_content': '**GOVERNMENT OF INDIA**',
 'chunk_index': 0}

In [4]:
chunks = [] 

In [10]:
some_var = chunks[-1]["chunk_id"] if chunks else "blae"
print(some_var)

blae
